In [111]:
import sqlite3
import pandas as pd

# Create a NEW database for DL prospects
conn = sqlite3.connect('nfl_analytics_dl.db')  # different name!
print("Connected to DL database")

# Your DL files
files = {
    'college_sacks': '../data/CollegeSacks13to25.csv',
    'college_tfls': '../data/CollegeTacklesForLoss13to25.csv',
    'college_tackles': '../data/CollegeTotalTackles13to25.csv',
    'combine': '../data/FullCombine13to26.csv',
    'nfl_stats': '../data/NFLStats18to25.csv',
    'tfl_ff': '../data/TFLsFFs18to25.csv'
}

# Load them the same way
for table_name, file_path in files.items():
    try:
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Loaded {table_name}: {len(df)} rows")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

conn.close()

Connected to DL database
Loaded college_sacks: 1652 rows
Loaded college_tfls: 1300 rows
Loaded college_tackles: 5453 rows
Loaded combine: 4730 rows
Loaded nfl_stats: 4000 rows
Loaded tfl_ff: 4100 rows


In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics_dl.db')

query = """
SELECT 
    n.Player as Name,
    c.Year as draft_year,
    c.Round as draft_round,
    c.PickOverall as overall_pick,

    -- Position classification
    CASE 
        -- Edge rushers
        WHEN n.Pos IN ('DE', 'LDE', 'RDE', 'DE/RDE', 'OLB', 'LOLB', 'ROLB', 'ROLB/LOLB', 'LOLB/ROLB', 'EDGE') THEN 'Edge'
        -- Interior DL
        WHEN n.Pos IN ('DT', 'NT', 'DL', 'DT/NG', 'DT/DE') THEN 'Interior'
        -- Catch any that don't fit (should be few)
        ELSE 'Other'
    END as position_group,

    
    -- Games and starts
    SUM(n.G) as total_games,
    SUM(n.GS) as games_started,
    
    -- Per game stats (pass rush)
    ROUND(CAST(SUM(n.Sk) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as sacks_per_game,
    ROUND(CAST(SUM(n.Prss) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as pressures_per_game,
    ROUND(CAST(SUM(n.QBKD) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as QBKDs_per_game,
    
    -- Per game stats (run defense)
    ROUND(CAST(SUM(t.TFL) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as tfl_per_game,
    ROUND(CAST(SUM(t.FF) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as ff_per_game,
    
    -- Career high stats (pass rush)
    MAX(n.Sk) as nfl_sacks_career_high,
    MAX(n.Prss) as nfl_pressures_career_high,
    MAX(n.QBKD) as nfl_QBKDs_career_high,
    MAX(n.Bats) as nfl_bats_career_high,
    MAX(n.Hrry) as nfl_hurries_career_high,
    
    -- Career high stats (run defense)
    MAX(t.TFL) as career_high_tfl,
    MAX(t.FF) as career_high_ff,
    
    -- Career totals
    SUM(n.Sk) as career_sacks,
    SUM(n.Prss) as career_pressures,
    SUM(t.TFL) as career_tfl,
    SUM(t.FF) as career_ff,
    
    -- Tackles (for context)
    SUM(t.SOLO) as career_solo_tackles,
    SUM(t.AST) as career_assist_tackles,
    ROUND(CAST(SUM(t.SOLO) AS FLOAT) / NULLIF(SUM(n.G), 0), 3) as solo_tackles_per_game

FROM nfl_stats n

-- Join combine data (draft info)
LEFT JOIN combine c ON n.Player = c.Player AND c.Pos IN ('DE', 'DT', 'DL', 'EDGE', 'OLB')  -- Exclude CB, S, WR, etc.

-- Join TFL/FF data
LEFT JOIN tfl_ff t ON n.Player = t.Name AND n.Year = t.Year

WHERE 
    n.Pos IN ('DE', 'DT', 'OLB', 'DL', 'LDE', 'RDE', 'LOLB', 'ROLB', 'DE/RDE', 'NT', 'DT/NG')
    AND n.G > 0  -- Players who actually played
    AND (n.Sk > 0 OR t.TFL > 0 OR t.FF > 0)  -- At least some production

GROUP BY 
    n.Player, c.Year, c.Round, c.PickOverall

ORDER BY 
    nfl_sacks_career_high DESC, 
    sacks_per_game DESC, 
    career_high_tfl DESC
"""
# LIMIT 50;

DL_summary = pd.read_sql_query(query, conn)

DL_summary.to_csv('dl_analysis_data.csv', index=False)

# print("TOP NFL PASS RUSHERS BY SACKS (with pressures as tiebreaker)")
# print("=" * 80)
# print(results.to_string(index=False))



conn.close()

# SUMMARY STATS

In [8]:
# Summary Stats
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")

# percentiles = [25, 50, 75, 90, 95, 99]
percentiles = [25, 50, 75, 90, 99]

# PER GAME STATS
print("\nPER GAME STATS")
print("\nSACKS PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['sacks_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} sacks")
print("=" * 80)

print("\nPRESSURES PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['pressures_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} pressures")
print("=" * 80)

print("\nQBKDs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['QBKDs_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} QBKDs")
print("=" * 80)

print("\nTFLs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['tfl_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} TFLs")
print("=" * 80)

print("\nFFs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['ff_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} FFs")
print("=" * 80)



# CAREER HIGH STATS
print("\nCAREER HIGH STATS")
print("\nCAREER HIGH SACKS PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['nfl_sacks_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} sacks")
print("=" * 80)

print("\nCAREER HIGH PRESSURE PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['nfl_pressures_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} pressures")
print("=" * 80)

print("\nCAREER HIGH QB KNOCKDOWN PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = DL_summary['nfl_QBKDs_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} QBKDs")
print("=" * 80)


###################################

# print("\nPLAYERS WITH 11 OR MORE SACKS (CAREER HIGH SEASON):")
# print("-" * 40)
elite_sacks = DL_summary[DL_summary['nfl_sacks_career_high'] >= 11]
# print(elite_sacks[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 11+ sacks: {len(elite_sacks)}")


# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_pressures = DL_summary[DL_summary['nfl_pressures_career_high'] >= 36]
# print(elite_pressures[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 36+ pressures: {len(elite_pressures)}")

# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_QBKDs = DL_summary[DL_summary['nfl_QBKDs_career_high'] >= 14]
# print(elite_QBKDs[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'nfl_QBKDs_career_high']].to_string(index=False))
#print(f"\nTotal players with 14+ QBKDs: {len(elite_QBKDs)}")



SUMMARY STATISTICS

PER GAME STATS

SACKS PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.094 sacks
50th percentile: 0.179 sacks
75th percentile: 0.324 sacks
90th percentile: 0.450 sacks
99th percentile: 0.820 sacks

PRESSURES PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.412 pressures
50th percentile: 0.705 pressures
75th percentile: 1.164 pressures
90th percentile: 1.673 pressures
99th percentile: 2.596 pressures

QBKDs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.126 QBKDs
50th percentile: 0.235 QBKDs
75th percentile: 0.400 QBKDs
90th percentile: 0.564 QBKDs
99th percentile: 1.000 QBKDs

TFLs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.225 TFLs
50th percentile: 0.333 TFLs
75th percentile: 0.500 TFLs
90th percentile: 0.695 TFLs
99th percentile: 1.114 TFLs

FFs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0

***
### Summary Stats Takeaways

**1. ** 

**2. ** 

**3. ** 

**The bottom line:** 

***

## Summary Stats for EDGE ONLY

In [119]:
# Summary Stats
print("\n" + "=" * 80)
print("EDGE SUMMARY STATISTICS")
edge_rushers = DL_summary[DL_summary['position_group'] == 'Edge']


# percentiles = [25, 50, 75, 90, 95, 99]
percentiles = [25, 50, 75, 90, 99]

# PER GAME STATS
print("\nPER GAME STATS")
print("\nSACKS PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['sacks_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} sacks")
print("=" * 80)

print("\nPRESSURES PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['pressures_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} pressures")
print("=" * 80)

print("\nQBKDs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['QBKDs_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} QBKDs")
print("=" * 80)

print("\nTFLs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['tfl_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} TFLs")
print("=" * 80)

print("\nFFs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['ff_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} FFs")
print("=" * 80)



# CAREER HIGH STATS
print("\nCAREER HIGH STATS")
print("\nCAREER HIGH SACKS PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['nfl_sacks_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} sacks")
print("=" * 80)

print("\nCAREER HIGH PRESSURE PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['nfl_pressures_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} pressures")
print("=" * 80)

print("\nCAREER HIGH QB KNOCKDOWN PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = edge_rushers['nfl_QBKDs_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} QBKDs")
print("=" * 80)


###################################

# print("\nPLAYERS WITH 11 OR MORE SACKS (CAREER HIGH SEASON):")
# print("-" * 40)
elite_sacks = edge_rushers[edge_rushers['nfl_sacks_career_high'] >= 11]
# print(elite_sacks[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 11+ sacks: {len(elite_sacks)}")


# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_pressures = edge_rushers[edge_rushers['nfl_pressures_career_high'] >= 36]
# print(elite_pressures[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 36+ pressures: {len(elite_pressures)}")

# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_QBKDs = edge_rushers[edge_rushers['nfl_QBKDs_career_high'] >= 14]
# print(elite_QBKDs[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'nfl_QBKDs_career_high']].to_string(index=False))
#print(f"\nTotal players with 14+ QBKDs: {len(elite_QBKDs)}")



EDGE SUMMARY STATISTICS

PER GAME STATS

SACKS PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.124 sacks
50th percentile: 0.231 sacks
75th percentile: 0.367 sacks
90th percentile: 0.505 sacks
99th percentile: 0.876 sacks

PRESSURES PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.585 pressures
50th percentile: 0.876 pressures
75th percentile: 1.378 pressures
90th percentile: 1.905 pressures
99th percentile: 2.837 pressures

QBKDs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.156 QBKDs
50th percentile: 0.291 QBKDs
75th percentile: 0.445 QBKDs
90th percentile: 0.614 QBKDs
99th percentile: 1.022 QBKDs

TFLs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.256 TFLs
50th percentile: 0.375 TFLs
75th percentile: 0.548 TFLs
90th percentile: 0.737 TFLs
99th percentile: 1.126 TFLs

FFs PER GAME PERCENTILES:
----------------------------------------
25th percenti

## Summary Stats for INTERIOR ONLY

In [6]:
# Summary Stats
print("\n" + "=" * 80)
print("INTERIOR SUMMARY STATISTICS")
interior_dl = DL_summary[DL_summary['position_group'] == 'Interior']


# percentiles = [25, 50, 75, 90, 95, 99]
percentiles = [25, 50, 75, 90, 99]

# PER GAME STATS
print("\nPER GAME STATS")
print("\nSACKS PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['sacks_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} sacks")
print("=" * 80)

print("\nPRESSURES PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['pressures_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} pressures")
print("=" * 80)

print("\nQBKDs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['QBKDs_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} QBKDs")
print("=" * 80)

print("\nTFLs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['tfl_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} TFLs")
print("=" * 80)

print("\nFFs PER GAME PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['ff_per_game'].quantile(p/100)
    print(f"{p}th percentile: {value:.3f} FFs")
print("=" * 80)



# CAREER HIGH STATS
print("\nCAREER HIGH STATS")
print("\nCAREER HIGH SACKS PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['nfl_sacks_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} sacks")
print("=" * 80)

print("\nCAREER HIGH PRESSURE PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['nfl_pressures_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} pressures")
print("=" * 80)

print("\nCAREER HIGH QB KNOCKDOWN PERCENTILES:")
print("-" * 40)
for p in percentiles:
    value = interior_dl['nfl_QBKDs_career_high'].quantile(p/100)
    print(f"{p}th percentile: {value:.1f} QBKDs")
print("=" * 80)


###################################

# print("\nPLAYERS WITH 11 OR MORE SACKS (CAREER HIGH SEASON):")
# print("-" * 40)
elite_sacks = interior_dl[interior_dl['nfl_sacks_career_high'] >= 11]
# print(elite_sacks[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 11+ sacks: {len(elite_sacks)}")


# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_pressures = interior_dl[interior_dl['nfl_pressures_career_high'] >= 36]
# print(elite_pressures[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'career_nfl_sacks']].to_string(index=False))
#print(f"\nTotal players with 36+ pressures: {len(elite_pressures)}")

# print("\nPLAYERS WITH 36 OR MORE PRESSURES (CAREER HIGH SEASON):")
# print("-" * 40)
elite_QBKDs = interior_dl[interior_dl['nfl_QBKDs_career_high'] >= 14]
# print(elite_QBKDs[['Name', 'nfl_sacks_career_high', 'nfl_pressures_career_high', 'nfl_QBKDs_career_high']].to_string(index=False))
#print(f"\nTotal players with 14+ QBKDs: {len(elite_QBKDs)}")



INTERIOR SUMMARY STATISTICS

PER GAME STATS

SACKS PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.063 sacks
50th percentile: 0.118 sacks
75th percentile: 0.200 sacks
90th percentile: 0.353 sacks
99th percentile: 0.643 sacks

PRESSURES PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.321 pressures
50th percentile: 0.458 pressures
75th percentile: 0.721 pressures
90th percentile: 1.152 pressures
99th percentile: 2.073 pressures

QBKDs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.110 QBKDs
50th percentile: 0.167 QBKDs
75th percentile: 0.274 QBKDs
90th percentile: 0.447 QBKDs
99th percentile: 0.863 QBKDs

TFLs PER GAME PERCENTILES:
----------------------------------------
25th percentile: 0.189 TFLs
50th percentile: 0.262 TFLs
75th percentile: 0.402 TFLs
90th percentile: 0.553 TFLs
99th percentile: 1.024 TFLs

FFs PER GAME PERCENTILES:
----------------------------------------
25th perc